<a href="https://colab.research.google.com/github/MukeshReddy8977/AI-COURSE-PROGRAMS-2026/blob/main/WEATHER_TRAVEL_AGENT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install groq requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 5.0 MB/s eta 0:00:00


In [2]:
import os
from getpass import getpass
from groq import Groq

GROQ_API_KEY = getpass("Enter your Groq API key: ")
client = Groq(api_key=GROQ_API_KEY)

print("Groq client initialized!")

Enter your Groq API key: ··········
Groq client initialized!


In [3]:
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role": "user", "content": "Reply with: Groq is working!"}
    ]
)
print(response.choices[0].message.content)

Groq is working!


In [4]:

import requests

def get_weather(location):
    try:
        # Find the city coordinates
        geo_url = "https://geocoding-api.open-meteo.com/v1/search"
        geo_response = requests.get(
            geo_url,
            params={"name": location, "count": 1, "language": "en", "format": "json"},
            timeout=15
        )
        geo_response.raise_for_status()
        places = geo_response.json().get("results", [])

        if not places:
            return {"error": f"City not found: {location}"}

        place = places[0]
        latitude = place["latitude"]
        longitude = place["longitude"]

        # Get current weather and today's forecast
        weather_url = "https://api.open-meteo.com/v1/forecast"
        weather_response = requests.get(
            weather_url,
            params={
                "latitude": latitude,
                "longitude": longitude,
                "current": "temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,weather_code,wind_speed_10m",
                "daily": "temperature_2m_max,temperature_2m_min,precipitation_probability_max",
                "forecast_days": 1,
                "timezone": "auto"
            },
            timeout=20
        )
        weather_response.raise_for_status()
        data = weather_response.json()

        return {
            "city": place["name"],
            "region": place.get("admin1", ""),
            "country": place.get("country", ""),
            "timezone": data.get("timezone"),
            "current": data.get("current", {}),
            "today": data.get("daily", {})
        }

    except requests.RequestException as e:
        return {"error": f"Weather service connection failed: {e}"}

# Test the replacement service
result = get_weather("Hyderabad")

if "error" in result:
    print(result["error"])
else:
    print("City:", result["city"], result["country"])
    print("Current weather:", result["current"])
    print("Today's forecast:", result["today"])


City: Hyderabad India
Current weather: {'time': '2026-10-19T17:30', 'interval': 900, 'temperature_2m': 30.7, 'relative_humidity_2m': 40, 'apparent_temperature': 30.8, 'precipitation': 0.0, 'weather_code': 2, 'wind_speed_10m': 12.8}
Today's forecast: {'time': ['2026-10-19'], 'temperature_2m_max': [32.7], 'temperature_2m_min': [24.0], 'precipitation_probability_max': [8]}


In [19]:

import sqlite3

conn = sqlite3.connect(
    "travel_agent.db",
    check_same_thread=False
)

cursor = conn.cursor()
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS preferences (
    city TEXT PRIMARY KEY,
    interests TEXT,
    travel_style TEXT
)
""")

conn.commit()
print("SQLite database is ready!")


SQLite database is ready!


In [20]:
import threading

db_lock = threading.Lock()

In [21]:
def save_preferences(city, interests, travel_style):
    with db_lock:
        conn.execute("""
            INSERT OR REPLACE INTO preferences
            (city, interests, travel_style)
            VALUES (?, ?, ?)
        """, (city.lower().strip(), interests, travel_style))
        conn.commit()

    return "Preferences saved successfully!"


def get_preferences(city):
    with db_lock:
        cursor = conn.execute(
            """SELECT interests, travel_style
               FROM preferences
               WHERE city = ?""",
            (city.lower().strip(),)
        )
        row = cursor.fetchone()

    if row:
        return {
            "interests": row[0],
            "travel_style": row[1]
        }

    return {
        "interests": "general sightseeing",
        "travel_style": "balanced"
    }

save_preferences(
    "Hyderabad",
    "historical places, food, photography",
    "budget-friendly"
)

print(get_preferences("Hyderabad"))

{'interests': 'historical places, food, photography', 'travel_style': 'budget-friendly'}


In [22]:

import json

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get real current weather and today's forecast for a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "City name, such as Hyderabad or Mumbai"
                    }
                },
                "required": ["location"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_preferences",
            "description": "Retrieve saved travel interests and travel style for a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "Destination city"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

available_functions = {
    "get_weather": get_weather,
    "get_preferences": get_preferences
}

print("Weather and preference tools are ready!")


Weather and preference tools are ready!


In [23]:
def run_agent(question):
    messages = [
        {
            "role": "system",
            "content": """
You are a professional Smart Weather and Travel Advisory Agent.

OBJECTIVES:
- Retrieve real weather data using the available weather tool.
- Retrieve saved travel preferences using the preferences tool.
- Generate clear, personalized and practical travel recommendations.

RESPONSE FORMAT:

## 1. Weather Summary
Report current temperature, feels-like temperature,
humidity, wind speed, precipitation and today's forecast
when these values are available from the tools.

## 2. Travel Suitability
Explain whether outdoor activities, indoor activities
or a combination is suitable based on actual conditions.

## 3. Recommended Itinerary
Provide a chronological morning, afternoon and evening plan.
Personalize suggestions using saved interests and travel style.
Do not invent attraction opening hours or travel times.

## 4. Weather Precautions
Give relevant, practical precautions based on the forecast.
Do not add irrelevant warnings.

## 5. Travel Tips
Give concise, useful suggestions for comfort and planning.

## 6. Final Recommendation
Finish with a short, decisive summary.

STRICT RULES:
1. Call the weather tool before making weather-dependent claims.
2. Call the preferences tool for the requested destination.
3. Never invent or guess weather readings.
4. Distinguish unavailable information from verified information.
5. Do not claim live prices, opening hours or transport
   availability unless verified by a reliable source.
6. If a tool fails, explain the limitation honestly.
7. Avoid repetition, generic filler and unnecessarily long answers.
8. Use readable headings, bullet points and simple language.
9. Never describe a forecast as a guarantee.
10. Base recommendations on retrieved data and user preferences.
"""
        },
        {"role": "user", "content": question}
    ]

    for attempt in range(5):
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=messages,
            tools=tools,
            tool_choice="auto",
            temperature=0.3
        )

        message = response.choices[0].message
        messages.append(message)

        if not message.tool_calls:
            return message.content or "I could not generate a response."

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name

            try:
                arguments = json.loads(tool_call.function.arguments)
                function = available_functions.get(function_name)

                if function is None:
                    output = {"error": "Unknown tool requested."}
                else:
                    output = function(**arguments)

            except Exception as e:
                output = {"error": str(e)}

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": function_name,
                "content": json.dumps(output, default=str)
            })

    return "The agent reached its tool-call limit. Please try again."

In [24]:
def run_agent(question):
    messages = [
        {
            "role": "system",
            "content": """
You are a Smart Weather and Travel Advisory Agent.
Use the weather tool for current weather or forecasts.
Use the preferences tool to retrieve saved travel interests.
Never invent weather data. If a tool fails, explain the problem.
Recommend activities based on the actual weather and preferences.
Consider rain, heat, wind and travel comfort.
Give practical advice, a suggested itinerary and useful precautions.
Do not claim an activity or attraction is open unless verified.
"""
        },
        {"role": "user", "content": question}
    ]

    for attempt in range(5):
        response = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=messages,
            tools=tools,
            tool_choice="auto",
            temperature=0.3
        )

        message = response.choices[0].message
        messages.append(message)

        if not message.tool_calls:
            return message.content or "I could not generate a response."

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name

            try:
                arguments = json.loads(tool_call.function.arguments)
                function = available_functions.get(function_name)

                if function is None:
                    output = {"error": "Unknown tool requested."}
                else:
                    output = function(**arguments)

            except Exception as e:
                output = {"error": str(e)}

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": function_name,
                "content": json.dumps(output, default=str)
            })

    return "The agent reached its tool-call limit. Please try again."

question = input("Where do you want to travel, and what advice do you need? ")
print("\n🤖 SMART TRAVEL ADVISOR\n")
print(run_agent(question))

Where do you want to travel, and what advice do you need? I'm planning a one-day trip to visakhapatnam. Check today's weather, use my saved preferences, suggest suitable activities, and give me a practical itinerary.

🤖 SMART TRAVEL ADVISOR

**Visakhapatnam – One‑Day Weather‑Smart Itinerary (19 Oct 2026)**  

| Time (local) | Activity | Why it fits today’s weather & your “balanced / general‑sightseeing” style |
|--------------|----------|-----------------------------------------------------------------------------|
| **07:00 – 08:30** | **Early‑morning walk on RK Beach** (sunrise, gentle sea breeze) | The forecast shows a *high chance of rain later* (≈ 91 % after midday) but the morning is still clear. A short beach stroll lets you enjoy the sea‑air and the iconic sunrise without getting soaked. |
| **08:45 – 09:15** | **Breakfast** at a beachfront café (e.g., *The Bay View*). | Light, energising meal; cafés open early and give you a chance to hydrate before the humidity climbs (≈ 80 %

In [25]:
!pip -q install fastapi uvicorn pyngrok python-multipart

In [26]:

from fastapi import FastAPI, Form
from fastapi.responses import HTMLResponse
from html import escape

app = FastAPI(title="Smart Weather and Travel Advisory Agent")

def make_page(result=""):
    return """
    <!DOCTYPE html>
    <html>
    <head>
        <meta name="viewport" content="width=device-width, initial-scale=1">
        <title>Smart Travel Advisor</title>
        <style>
            body {
                font-family: Arial, sans-serif;
                background: linear-gradient(135deg, #dbeafe, #f0fdf4);
                margin: 0;
                padding: 25px;
                color: #172554;
            }
            .container {
                max-width: 750px;
                margin: 35px auto;
                background: white;
                padding: 30px;
                border-radius: 18px;
                box-shadow: 0 8px 30px #0001;
            }
            h1 { color: #1d4ed8; }
            p { line-height: 1.6; }
            input, textarea {
                box-sizing: border-box;
                width: 100%;
                padding: 13px;
                margin: 8px 0 18px;
                border: 1px solid #cbd5e1;
                border-radius: 9px;
                font-size: 16px;
            }
            button {
                background: #2563eb;
                color: white;
                padding: 14px;
                width: 100%;
                border: 0;
                border-radius: 9px;
                font-size: 16px;
                cursor: pointer;
            }
            button:hover { background: #1d4ed8; }
            .result {
                margin-top: 25px;
                padding: 18px;
                background: #eff6ff;
                border-radius: 10px;
                white-space: pre-wrap;
                overflow-wrap: anywhere;
                line-height: 1.7;
            }
            .footer {
                margin-top: 25px;
                color: #64748b;
                font-size: 13px;
            }
        </style>
    </head>
    <body>
        <div class="container">
            <h1>🌦️ Smart Weather & Travel Advisor</h1>
            <p>Plan smarter trips with live weather information and AI-powered travel suggestions.</p>
            <form action="/ask" method="post">
                <label>Destination</label>
                <input name="city" placeholder="e.g. Hyderabad" required>
                <label>Your travel question</label>
                <textarea name="question" rows="4"
                placeholder="Plan a one-day trip based on today's weather..."
                required></textarea>
                <button type="submit">Get Travel Advice ✈️</button>
            </form>
            RESULT_PLACEHOLDER
            <div class="footer">
                Powered by Groq AI, Open-Meteo and SQLite
            </div>
        </div>
    </body>
    </html>
    """.replace("RESULT_PLACEHOLDER", result)

@app.get("/", response_class=HTMLResponse)
def home():
    return make_page()

@app.post("/ask", response_class=HTMLResponse)
def ask(city: str = Form(...), question: str = Form(...)):
    prompt = (
        f"Destination: {city.strip()}. "
        f"Travel request: {question.strip()}. "
        "Check the actual weather and saved preferences for this destination. "
        "Give a practical travel itinerary based on the retrieved data."
    )
    try:
        answer = run_agent(prompt)
        result_html = (
            '<div class="result"><h2>✈️ Your Travel Advice</h2><p>'
            + escape(str(answer)) +
            '</p></div>'
        )
    except Exception as e:
        result_html = (
            '<div class="result"><h2>Something went wrong</h2><p>'
            + escape(str(e)) + '</p></div>'
        )
    return make_page(result_html)

print("FastAPI webpage is ready!")


FastAPI webpage is ready!


In [27]:
from getpass import getpass
from pyngrok import ngrok
import threading
import time
import uvicorn

NGROK_TOKEN = getpass("Enter your ngrok authtoken:")
ngrok.set_auth_token(NGROK_TOKEN)

# Start the FastAPI server
def start_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()

# Wait for the server to start
time.sleep(3)

# Create a public HTTPS tunnel
tunnel = ngrok.connect(8000, "http")
print("YOUR WEBSITE URL:", tunnel.public_url)


Enter your ngrok authtoken:··········


ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use


YOUR WEBSITE URL: https://smoking-kilogram-plop.ngrok-free.dev
